# Prompt Engineering with Groq API LLMs — Hands-on Colab Exercise

This notebook covers:
- Zero-shot, one-shot, few-shot
- Role/persona prompting
- Instruction prompting
- Context-grounded prompting
- JSON / structured output prompting
- Classification, extraction, summarization, rewriting
- Reasoning-style prompting, step-back prompting
- Self-critique, prompt chaining, multi-turn prompting
- Rubric-based evaluation and prompt templates

In [1]:
print("Prompt Engineering with Groq API LLMs - Hands-on Colab Exercise")
print("We will test multiple prompt engineering techniques cell by cell.")

Prompt Engineering with Groq API LLMs - Hands-on Colab Exercise
We will test multiple prompt engineering techniques cell by cell.


In [2]:
!pip install -q groq pandas tabulate

In [3]:
import os
import json
import textwrap
import pandas as pd
from groq import Groq
from tabulate import tabulate

In [6]:
from dotenv import load_dotenv

load_dotenv()
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)


In [7]:
MODEL_NAME = "llama-3.3-70b-versatile"
print("Using model:", MODEL_NAME)

Using model: llama-3.3-70b-versatile


In [8]:
def ask_groq(prompt, system_prompt="You are a helpful AI assistant.", model=MODEL_NAME, temperature=0.3, max_tokens=1024):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [9]:
def show_result(title, prompt, response, width=110):
    print("="*120)
    print(f"TECHNIQUE: {title}")
    print("-"*120)
    print("PROMPT:")
    print(textwrap.fill(prompt, width=width))
    print("-"*120)
    print("MODEL RESPONSE:")
    print(response)
    print("="*120)

In [10]:
customer_feedback = """
I liked the app interface and the speed of checkout. However, the payment page failed twice before succeeding.
Customer support replied after 2 days, which is too slow. Product quality was good overall, but delivery packaging was damaged.
I may still recommend it because the discounts were attractive.
"""
print(customer_feedback)


I liked the app interface and the speed of checkout. However, the payment page failed twice before succeeding.
Customer support replied after 2 days, which is too slow. Product quality was good overall, but delivery packaging was damaged.
I may still recommend it because the discounts were attractive.



In [11]:
data = [
    {"feedback_id": 101, "feedback": "Checkout was fast and the UI is clean, but payment failed once and support was slow."},
    {"feedback_id": 102, "feedback": "Good product quality, nice discount, but damaged packaging and late delivery."},
    {"feedback_id": 103, "feedback": "The app is intuitive and easy to use. Customer support resolved my issue quickly."}
]
df = pd.DataFrame(data)
df

,feedback_id,feedback
0,101,"Checkout was fast and the UI is clean, but pay..."
1,102,"Good product quality, nice discount, but damag..."
2,103,The app is intuitive and easy to use. Customer...


In [12]:
prompt_zero_shot = f"""
Read the customer feedback below and identify:
1. Positive points
2. Negative points
3. Final customer sentiment as Positive / Neutral / Negative

Feedback:
{customer_feedback}
"""
response_zero_shot = ask_groq(prompt_zero_shot)
show_result("Zero-shot Prompting", prompt_zero_shot, response_zero_shot)

TECHNIQUE: Zero-shot Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Read the customer feedback below and identify: 1. Positive points 2. Negative points 3. Final customer
sentiment as Positive / Neutral / Negative  Feedback:  I liked the app interface and the speed of checkout.
However, the payment page failed twice before succeeding. Customer support replied after 2 days, which is too
slow. Product quality was good overall, but delivery packaging was damaged. I may still recommend it because
the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Based on the customer feedback, here are the identified points:

1. **Positive points**:
   - The app interface was liked by the customer.
   - The speed of checkout was satisfactory.
   - Product quality was good overall.
   - Discounts wer

In [13]:
prompt_one_shot = f"""
Example:
Feedback: "The app design is nice but delivery was late."
Output:
Positive points: app design
Negative points: late delivery
Sentiment: Neutral

Now do the same for the feedback below.

Feedback:
{customer_feedback}
"""
response_one_shot = ask_groq(prompt_one_shot)
show_result("One-shot Prompting", prompt_one_shot, response_one_shot)

TECHNIQUE: One-shot Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Example: Feedback: "The app design is nice but delivery was late." Output: Positive points: app design
Negative points: late delivery Sentiment: Neutral  Now do the same for the feedback below.  Feedback:  I liked
the app interface and the speed of checkout. However, the payment page failed twice before succeeding.
Customer support replied after 2 days, which is too slow. Product quality was good overall, but delivery
packaging was damaged. I may still recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here's the analysis of the feedback:

Positive points: 
- app interface
- speed of checkout
- product quality
- attractive discounts

Negative points: 
- payment page failed twice
- slow custome

In [14]:
prompt_few_shot = f"""
You are analyzing e-commerce customer feedback.

Example 1:
Feedback: "The product quality is excellent but shipping was delayed."
Output:
Positive points: product quality
Negative points: delayed shipping
Sentiment: Neutral

Example 2:
Feedback: "Support was quick, refund was smooth, and the app was easy to use."
Output:
Positive points: quick support, smooth refund, easy app usage
Negative points: none
Sentiment: Positive

Example 3:
Feedback: "Payment failed twice and the package arrived broken."
Output:
Positive points: none
Negative points: payment failure, damaged package
Sentiment: Negative

Now analyze this feedback:

Feedback:
{customer_feedback}
"""
response_few_shot = ask_groq(prompt_few_shot)
show_result("Few-shot Prompting", prompt_few_shot, response_few_shot)

TECHNIQUE: Few-shot Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 You are analyzing e-commerce customer feedback.  Example 1: Feedback: "The product quality is excellent but
shipping was delayed." Output: Positive points: product quality Negative points: delayed shipping Sentiment:
Neutral  Example 2: Feedback: "Support was quick, refund was smooth, and the app was easy to use." Output:
Positive points: quick support, smooth refund, easy app usage Negative points: none Sentiment: Positive
Example 3: Feedback: "Payment failed twice and the package arrived broken." Output: Positive points: none
Negative points: payment failure, damaged package Sentiment: Negative  Now analyze this feedback:  Feedback:
I liked the app interface and the speed of checkout. However, the payment page failed twice before succeeding.
Customer support replied after 2 days, which is too slow. Product quality was good ove

In [15]:
prompt_role = f"""
You are a senior Customer Experience Analyst in an e-commerce company.
Analyze the feedback and provide:
1. Customer pain points
2. Business risks
3. Improvement recommendations

Feedback:
{customer_feedback}
"""
response_role = ask_groq(prompt_role)
show_result("Role / Persona Prompting", prompt_role, response_role)

TECHNIQUE: Role / Persona Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 You are a senior Customer Experience Analyst in an e-commerce company. Analyze the feedback and provide: 1.
Customer pain points 2. Business risks 3. Improvement recommendations  Feedback:  I liked the app interface
and the speed of checkout. However, the payment page failed twice before succeeding. Customer support replied
after 2 days, which is too slow. Product quality was good overall, but delivery packaging was damaged. I may
still recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
**Analysis of Customer Feedback**

### 1. Customer Pain Points:

- **Technical Issues:** The customer experienced difficulties with the payment page, which failed twice before the transaction was successf

In [16]:
prompt_instruction = f"""
Task: Analyze the feedback.

Rules:
- Use bullet points
- Keep each bullet under 15 words
- Separate positives, negatives, sentiment, and action items
- Do not add information not present in the feedback

Feedback:
{customer_feedback}
"""
response_instruction = ask_groq(prompt_instruction)
show_result("Instruction-based Prompting", prompt_instruction, response_instruction)

TECHNIQUE: Instruction-based Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Task: Analyze the feedback.  Rules: - Use bullet points - Keep each bullet under 15 words - Separate
positives, negatives, sentiment, and action items - Do not add information not present in the feedback
Feedback:  I liked the app interface and the speed of checkout. However, the payment page failed twice before
succeeding. Customer support replied after 2 days, which is too slow. Product quality was good overall, but
delivery packaging was damaged. I may still recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
**Positives:**
* Liked app interface
* Fast checkout
* Good product quality
* Attractive discounts

**Negatives:**
* Payment page failed
* Slow support reply
* Damaged packagin

In [17]:
prompt_context = f"""
Context:
You are working for an e-commerce platform whose priorities are:
1. Reduce payment failures
2. Improve customer support turnaround time
3. Reduce packaging damage complaints

Task:
Analyze the feedback in the context of these business priorities and rank the top 3 issues.

Feedback:
{customer_feedback}
"""
response_context = ask_groq(prompt_context)
show_result("Context-grounded Prompting", prompt_context, response_context)

TECHNIQUE: Context-grounded Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Context: You are working for an e-commerce platform whose priorities are: 1. Reduce payment failures 2.
Improve customer support turnaround time 3. Reduce packaging damage complaints  Task: Analyze the feedback in
the context of these business priorities and rank the top 3 issues.  Feedback:  I liked the app interface and
the speed of checkout. However, the payment page failed twice before succeeding. Customer support replied
after 2 days, which is too slow. Product quality was good overall, but delivery packaging was damaged. I may
still recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Based on the provided feedback, I've analyzed the issues in the context of the e-commerce platform

In [19]:
prompt_json = f"""
Analyze the customer feedback and return output in valid JSON only.

Required schema:
{{
  "positive_points": [],
  "negative_points": [],
  "sentiment": "",
  "recommendation_priority": []
}}

Customer Feedback:
{customer_feedback}
"""
response_json = ask_groq(prompt_json)
show_result("Structured / JSON Output Prompting", prompt_json, response_json)

TECHNIQUE: Structured / JSON Output Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Analyze the customer feedback and return output in valid JSON only.  Required schema: {   "positive_points":
[],   "negative_points": [],   "sentiment": "",   "recommendation_priority": [] }  Customer Feedback:  I liked
the app interface and the speed of checkout. However, the payment page failed twice before succeeding.
Customer support replied after 2 days, which is too slow. Product quality was good overall, but delivery
packaging was damaged. I may still recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
```
{
  "positive_points": [
    "app interface",
    "speed of checkout",
    "product quality",
    "discounts"
  ],
  "negative_points": [
    "payment page issues",
   

In [20]:
try:
    parsed = json.loads(response_json)
    print(json.dumps(parsed, indent=2))
except Exception as e:
    print("JSON parsing failed:", e)
    print("Raw output:")
    print(response_json)

JSON parsing failed: Expecting value: line 1 column 1 (char 0)
Raw output:
```
{
  "positive_points": [
    "app interface",
    "speed of checkout",
    "product quality",
    "discounts"
  ],
  "negative_points": [
    "payment page issues",
    "slow customer support response",
    "damaged delivery packaging"
  ],
  "sentiment": "neutral",
  "recommendation_priority": [
    "improve payment page functionality",
    "enhance customer support response time",
    "ensure proper delivery packaging"
  ]
}
```


In [21]:
prompt_classification = f"""
Classify the feedback into one of these categories:
- Product Issue
- Delivery Issue
- Support Issue
- Payment Issue
- UX/UI Praise
- Mixed Experience

Return:
1. Primary category
2. Secondary category if any
3. Reason in 2 lines

Feedback:
{customer_feedback}
"""
response_classification = ask_groq(prompt_classification)
show_result("Classification Prompting", prompt_classification, response_classification)

TECHNIQUE: Classification Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Classify the feedback into one of these categories: - Product Issue - Delivery Issue - Support Issue -
Payment Issue - UX/UI Praise - Mixed Experience  Return: 1. Primary category 2. Secondary category if any 3.
Reason in 2 lines  Feedback:  I liked the app interface and the speed of checkout. However, the payment page
failed twice before succeeding. Customer support replied after 2 days, which is too slow. Product quality was
good overall, but delivery packaging was damaged. I may still recommend it because the discounts were
attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
1. Mixed Experience
2. Payment Issue, Support Issue, Delivery Issue 
3. The customer had both positive and negative experiences, including issues with p

In [22]:
prompt_extraction = f"""
Extract the following from the feedback:
- product_quality_issue: yes/no
- delivery_issue: yes/no
- support_issue: yes/no
- payment_issue: yes/no
- discount_mentioned: yes/no
- recommendation_intent: yes/no

Feedback:
{customer_feedback}

Return only JSON.
"""
response_extraction = ask_groq(prompt_extraction)
show_result("Information Extraction Prompting", prompt_extraction, response_extraction)

TECHNIQUE: Information Extraction Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Extract the following from the feedback: - product_quality_issue: yes/no - delivery_issue: yes/no -
support_issue: yes/no - payment_issue: yes/no - discount_mentioned: yes/no - recommendation_intent: yes/no
Feedback:  I liked the app interface and the speed of checkout. However, the payment page failed twice before
succeeding. Customer support replied after 2 days, which is too slow. Product quality was good overall, but
delivery packaging was damaged. I may still recommend it because the discounts were attractive.   Return only
JSON.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
```json
{
  "product_quality_issue": "no",
  "delivery_issue": "yes",
  "support_issue": "yes",
  "payment_issue": "yes",
  "discount_mentioned": "y

In [23]:
prompt_summary = f"""
Summarize the feedback in:
1. 1 sentence
2. 3 bullet points
3. an executive summary under 50 words

Feedback:
{customer_feedback}
"""
response_summary = ask_groq(prompt_summary)
show_result("Summarization Prompting", prompt_summary, response_summary)

TECHNIQUE: Summarization Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Summarize the feedback in: 1. 1 sentence 2. 3 bullet points 3. an executive summary under 50 words  Feedback:
I liked the app interface and the speed of checkout. However, the payment page failed twice before succeeding.
Customer support replied after 2 days, which is too slow. Product quality was good overall, but delivery
packaging was damaged. I may still recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here are the summaries:

1. The customer had a mixed experience with the app, citing both positive and negative aspects, including a faulty payment page and slow customer support.
2. Here are the key points:
* The app interface and checkout speed were satisfactory
* The payment page e

In [24]:
prompt_rewrite = f"""
Rewrite the following feedback into:
1. A professional escalation email to the operations team
2. A concise CRM note
3. A tweet-sized summary

Feedback:
{customer_feedback}
"""
response_rewrite = ask_groq(prompt_rewrite)
show_result("Transformation / Rewriting Prompting", prompt_rewrite, response_rewrite)

TECHNIQUE: Transformation / Rewriting Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Rewrite the following feedback into: 1. A professional escalation email to the operations team 2. A concise
CRM note 3. A tweet-sized summary  Feedback:  I liked the app interface and the speed of checkout. However,
the payment page failed twice before succeeding. Customer support replied after 2 days, which is too slow.
Product quality was good overall, but delivery packaging was damaged. I may still recommend it because the
discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here are the rewritten feedback in the requested formats:

**1. Professional Escalation Email to the Operations Team:**

Subject: Escalation: Payment Page Issues, Delayed Customer Support, and Damaged Packaging

Dear Operations 

In [25]:
prompt_reasoning = f"""
Analyze the customer feedback step by step.

Return:
1. Key observations
2. How you inferred sentiment
3. Final conclusion

Keep reasoning concise and task-focused.

Feedback:
{customer_feedback}
"""
response_reasoning = ask_groq(prompt_reasoning)
show_result("Reasoning-style Prompting", prompt_reasoning, response_reasoning)

TECHNIQUE: Reasoning-style Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Analyze the customer feedback step by step.  Return: 1. Key observations 2. How you inferred sentiment 3.
Final conclusion  Keep reasoning concise and task-focused.  Feedback:  I liked the app interface and the speed
of checkout. However, the payment page failed twice before succeeding. Customer support replied after 2 days,
which is too slow. Product quality was good overall, but delivery packaging was damaged. I may still recommend
it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here's the analysis of the customer feedback:

1. **Key observations**: 
   - The customer liked the app interface and checkout speed.
   - The payment page had issues, failing twice before succeeding.
   - Customer 

In [26]:
prompt_step_back = f"""
Before analyzing the feedback, first answer:
"What are the common dimensions used to evaluate customer feedback in e-commerce?"

Then use those dimensions to analyze the following feedback:
{customer_feedback}

Return:
1. General evaluation dimensions
2. Feedback analysis using those dimensions
"""
response_step_back = ask_groq(prompt_step_back)
show_result("Step-back Prompting", prompt_step_back, response_step_back)

TECHNIQUE: Step-back Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Before analyzing the feedback, first answer: "What are the common dimensions used to evaluate customer
feedback in e-commerce?"  Then use those dimensions to analyze the following feedback:  I liked the app
interface and the speed of checkout. However, the payment page failed twice before succeeding. Customer
support replied after 2 days, which is too slow. Product quality was good overall, but delivery packaging was
damaged. I may still recommend it because the discounts were attractive.   Return: 1. General evaluation
dimensions 2. Feedback analysis using those dimensions
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
**1. General evaluation dimensions:**
The common dimensions used to evaluate customer feedback in e-commerce include:
- *

In [27]:
initial_prompt = f"""
Analyze the feedback and provide positives, negatives, sentiment, and action items.

Feedback:
{customer_feedback}
"""
initial_response = ask_groq(initial_prompt)

critique_prompt = f"""
You previously generated the following analysis:

{initial_response}

Now critique it for:
1. Missing points
2. Overstatements
3. Weak recommendations
4. Better final version

Original feedback:
{customer_feedback}
"""
critique_response = ask_groq(critique_prompt)
show_result("Initial Analysis", initial_prompt, initial_response)
show_result("Self-reflection / Critique Prompting", critique_prompt, critique_response)

TECHNIQUE: Initial Analysis
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Analyze the feedback and provide positives, negatives, sentiment, and action items.  Feedback:  I liked the
app interface and the speed of checkout. However, the payment page failed twice before succeeding. Customer
support replied after 2 days, which is too slow. Product quality was good overall, but delivery packaging was
damaged. I may still recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
**Analysis of Feedback**

**Positives:**

1. The app interface was liked by the customer, indicating a user-friendly design.
2. The speed of checkout was appreciated, suggesting a smooth and efficient payment process.
3. The product quality was good overall, indicating that the customer was satisfied with

In [28]:
prompt_chain_1 = f"""
Extract all issues mentioned in the feedback as a bullet list.

Feedback:
{customer_feedback}
"""
issues_output = ask_groq(prompt_chain_1)
show_result("Prompt Chaining - Stage 1", prompt_chain_1, issues_output)

TECHNIQUE: Prompt Chaining - Stage 1
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Extract all issues mentioned in the feedback as a bullet list.  Feedback:  I liked the app interface and the
speed of checkout. However, the payment page failed twice before succeeding. Customer support replied after 2
days, which is too slow. Product quality was good overall, but delivery packaging was damaged. I may still
recommend it because the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here are the issues mentioned in the feedback as a bullet list:

* The payment page failed twice before succeeding
* Customer support replied too slowly (after 2 days)
* Delivery packaging was damaged


In [29]:
prompt_chain_2 = f"""
Given these extracted issues:
{issues_output}

Rank them by business severity from highest to lowest.
Explain each ranking in one line.
"""
priority_output = ask_groq(prompt_chain_2)
show_result("Prompt Chaining - Stage 2", prompt_chain_2, priority_output)

TECHNIQUE: Prompt Chaining - Stage 2
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Given these extracted issues: Here are the issues mentioned in the feedback as a bullet list:  * The payment
page failed twice before succeeding * Customer support replied too slowly (after 2 days) * Delivery packaging
was damaged  Rank them by business severity from highest to lowest. Explain each ranking in one line.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here are the issues ranked by business severity from highest to lowest:
1. The payment page failed twice before succeeding - This issue has the highest severity as it directly affects revenue and can lead to lost sales.
2. Customer support replied too slowly (after 2 days) - This issue has moderate severity as delayed support can erode customer trust and loyalty.
3. Delive

In [30]:
prompt_chain_3 = f"""
Using this prioritized issue list:
{priority_output}

Create a 30-day action plan for the operations team.
"""
action_plan_output = ask_groq(prompt_chain_3)
show_result("Prompt Chaining - Stage 3", prompt_chain_3, action_plan_output)

TECHNIQUE: Prompt Chaining - Stage 3
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Using this prioritized issue list: Here are the issues ranked by business severity from highest to lowest: 1.
The payment page failed twice before succeeding - This issue has the highest severity as it directly affects
revenue and can lead to lost sales. 2. Customer support replied too slowly (after 2 days) - This issue has
moderate severity as delayed support can erode customer trust and loyalty. 3. Delivery packaging was damaged -
This issue has the lowest severity as it is a relatively minor problem that can be easily resolved through
replacement or refund, without significant long-term impact.  Create a 30-day action plan for the operations
team.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
**30-Day Action Plan for Operations T

In [31]:
def chat_groq(messages, model=MODEL_NAME, temperature=0.3, max_tokens=1024):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=messages
    )
    return response.choices[0].message.content

In [32]:
messages = [
    {"role": "system", "content": "You are a customer feedback analyst."},
    {"role": "user", "content": f"Analyze this feedback:\n{customer_feedback}"}
]

resp1 = chat_groq(messages)
print("Assistant:", resp1)

messages.append({"role": "assistant", "content": resp1})
messages.append({"role": "user", "content": "Now convert that into a priority matrix with High / Medium / Low severity."})

resp2 = chat_groq(messages)
print("\nAssistant Follow-up:\n", resp2)

Assistant: **Summary:**
The customer had a mixed experience with the app. They liked the interface and checkout speed, but encountered issues with the payment page and customer support response time. The product quality was good, but the delivery packaging was damaged. Despite these issues, the customer might still recommend the app due to attractive discounts.

**Key Strengths:**

1. **App Interface**: The customer liked the app's interface, indicating a well-designed and user-friendly experience.
2. **Checkout Speed**: The checkout process was fast, which is a significant factor in ensuring a smooth customer experience.
3. **Product Quality**: The customer was satisfied with the overall quality of the product.
4. **Discounts**: The attractive discounts offered by the app were a major draw for the customer.

**Key Areas for Improvement:**

1. **Payment Page**: The payment page failed twice before succeeding, which can be frustrating and may lead to abandoned transactions.
2. **Custome

In [33]:
candidate_answer = """
Positive points: app UI, checkout speed, discounts
Negative points: payment page failed, support was slow, packaging damaged
Sentiment: Neutral
"""

prompt_eval = f"""
Evaluate the following answer on a scale of 1 to 5 for each criterion:
- Completeness
- Accuracy
- Relevance
- Clarity

Also provide:
- total score /20
- 3 improvement suggestions

Original feedback:
{customer_feedback}

Candidate answer:
{candidate_answer}
"""
response_eval = ask_groq(prompt_eval)
show_result("Rubric-based Evaluation Prompting", prompt_eval, response_eval)

TECHNIQUE: Rubric-based Evaluation Prompting
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 Evaluate the following answer on a scale of 1 to 5 for each criterion: - Completeness - Accuracy - Relevance
- Clarity  Also provide: - total score /20 - 3 improvement suggestions  Original feedback:  I liked the app
interface and the speed of checkout. However, the payment page failed twice before succeeding. Customer
support replied after 2 days, which is too slow. Product quality was good overall, but delivery packaging was
damaged. I may still recommend it because the discounts were attractive.   Candidate answer:  Positive points:
app UI, checkout speed, discounts Negative points: payment page failed, support was slow, packaging damaged
Sentiment: Neutral
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here's the evaluati

In [34]:
def build_feedback_prompt(feedback, task_type="summary"):
    templates = {
        "summary": f"Summarize the following feedback in 3 bullet points:\n{feedback}",
        "sentiment": f"Identify sentiment and explain why:\n{feedback}",
        "action_items": f"Read the feedback and propose action items:\n{feedback}",
        "json_extract": f"""
Extract issues from this feedback and return JSON with:
payment_issue, support_issue, delivery_issue, packaging_issue, sentiment

Feedback:
{feedback}
"""
    }
    return templates.get(task_type, templates["summary"])

In [35]:
for task in ["summary", "sentiment", "action_items", "json_extract"]:
    p = build_feedback_prompt(customer_feedback, task)
    r = ask_groq(p)
    show_result(f"Prompt Template: {task}", p, r)

TECHNIQUE: Prompt Template: summary
------------------------------------------------------------------------------------------------------------------------
PROMPT:
Summarize the following feedback in 3 bullet points:  I liked the app interface and the speed of checkout.
However, the payment page failed twice before succeeding. Customer support replied after 2 days, which is too
slow. Product quality was good overall, but delivery packaging was damaged. I may still recommend it because
the discounts were attractive.
------------------------------------------------------------------------------------------------------------------------
MODEL RESPONSE:
Here are 3 bullet points summarizing the feedback:

* The app interface was liked, and checkout was fast, but the payment page experienced technical issues.
* Customer support response time was slow, taking 2 days to reply, and the delivery packaging was damaged.
* Despite some issues, the customer might still recommend the app due to attr

In [36]:
comparison_prompts = {
    "zero_shot": f"Analyze the sentiment of this feedback:\n{customer_feedback}",
    "role_based": f"You are a CX manager. Analyze sentiment and business risks:\n{customer_feedback}",
    "json_output": f"""
Return JSON with sentiment, positives, negatives for:
{customer_feedback}
""",
    "few_shot": f"""
Example:
Feedback: Delivery was late but the product was good.
Sentiment: Neutral

Now analyze:
{customer_feedback}
"""
}

results = []
for name, pr in comparison_prompts.items():
    out = ask_groq(pr)
    results.append({"prompt_type": name, "response_preview": out[:300]})

comparison_df = pd.DataFrame(results)
comparison_df

,prompt_type,response_preview
0,zero_shot,"The sentiment of this feedback is mixed, but o..."
1,role_based,**Sentiment Analysis:**\nThe customer's sentim...
2,json_output,"Here is the JSON output with sentiment, positi..."
3,few_shot,Feedback: The customer had mixed experiences w...


In [37]:
capstone_text = """
A retail company has collected 500 customer reviews from app, email, and support tickets.
Management wants:
1. top complaint categories,
2. overall sentiment,
3. urgent operational issues,
4. a concise weekly CX report.
"""
print(capstone_text)


A retail company has collected 500 customer reviews from app, email, and support tickets.
Management wants:
1. top complaint categories,
2. overall sentiment,
3. urgent operational issues,
4. a concise weekly CX report.



In [38]:
prompt_capstone = f"""
You are a Lead Customer Experience Analyst.

Problem:
{capstone_text}

Using the sample feedback below as if it were part of the weekly dataset:
{customer_feedback}

Prepare a weekly CX insight report with:
1. Top issues
2. Positive themes
3. Risk areas
4. Immediate actions
5. Suggested dashboard KPIs
"""
response_capstone = ask_groq(prompt_capstone)
show_result("Capstone Prompt Engineering Exercise", prompt_capstone, response_capstone)

TECHNIQUE: Capstone Prompt Engineering Exercise
------------------------------------------------------------------------------------------------------------------------
PROMPT:
 You are a Lead Customer Experience Analyst.  Problem:  A retail company has collected 500 customer reviews
from app, email, and support tickets. Management wants: 1. top complaint categories, 2. overall sentiment, 3.
urgent operational issues, 4. a concise weekly CX report.   Using the sample feedback below as if it were part
of the weekly dataset:  I liked the app interface and the speed of checkout. However, the payment page failed
twice before succeeding. Customer support replied after 2 days, which is too slow. Product quality was good
overall, but delivery packaging was damaged. I may still recommend it because the discounts were attractive.
Prepare a weekly CX insight report with: 1. Top issues 2. Positive themes 3. Risk areas 4. Immediate actions
5. Suggested dashboard KPIs
------------------------------

In [39]:
def analyze_feedback_row(feedback):
    prompt = f"""
Analyze this customer feedback and return JSON with:
- sentiment
- positives
- negatives
- main_issue_category

Feedback:
{feedback}
"""
    return ask_groq(prompt)

df["llm_analysis"] = df["feedback"].apply(analyze_feedback_row)
df

,feedback_id,feedback,llm_analysis
0,101,"Checkout was fast and the UI is clean, but pay...","```json\n{\n ""sentiment"": ""neutral"",\n ""posi..."
1,102,"Good product quality, nice discount, but damag...",Here's the analysis of the customer feedback i...
2,103,The app is intuitive and easy to use. Customer...,"```json\n{\n ""sentiment"": ""positive"",\n ""pos..."


In [40]:
reflection_questions = [
    "Which prompting technique gave the most structured output?",
    "Which prompt reduced hallucination risk the most?",
    "Which technique is best for analytics pipelines?",
    "Which technique is best for chatbots?",
    "How did output quality change from zero-shot to few-shot?",
    "When should prompt chaining be preferred over one big prompt?"
]

for i, q in enumerate(reflection_questions, 1):
    print(f"{i}. {q}")

1. Which prompting technique gave the most structured output?
2. Which prompt reduced hallucination risk the most?
3. Which technique is best for analytics pipelines?
4. Which technique is best for chatbots?
5. How did output quality change from zero-shot to few-shot?
6. When should prompt chaining be preferred over one big prompt?
